# FairSpeech: ASR Inference & Feedback Gate Pipeline (Member B - WP2)

**Lead:** Sanjay Eshwaradurai Vasantthisri (ST93197)

This notebook demonstrates:
1. **ASR Inference** (Whisper, CrisperWhisper, Vosk) with word-level timestamps and confidence scores.
2. **Word Error Rate (WER)** evaluation and analysis of confident errors and hallucinations by accent cluster.
3. **Feedback Gate Feature Extraction** (confidence context window, 3-way agreement, Zipf frequency, speech rate).
4. **Baseline & Gate Classifier Training** (Logistic Regression & LightGBM with grouped 5-fold CV).
5. **Evaluation**: Genuine recall >= 90% constraint, Leave-One-Accent-Out cross-validation, speaker-level bootstrap CIs, and SHAP analysis.

In [ ]:
# 1. Install dependencies for GPU execution on Google Colab
# !pip install -q -r requirements.txt
# !pip install -q -r requirements-asr.txt
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Imports
import pandas as pd
import numpy as np
from src.asr import normalise_text, compute_wer, compute_group_wer, transcribe_multi
from src.gate import (
    build_features,
    ConfidenceThresholdBaseline,
    train_gate,
    evaluate_gate,
    evaluate_leave_one_accent_out,
    speaker_bootstrap_ci,
    run_shap_analysis
)

In [ ]:
# 3. Load Gold Dataset from Member A
gold_df = pd.read_csv("data/gold_dataset.csv")
print("Loaded gold dataset:", gold_df.shape)
gold_df.head(5)

In [ ]:
# 4. ASR Transcription & Word Error Rate (WER) Evaluation
# Transcribing segments across Whisper, CrisperWhisper, and Vosk
sample_turn = gold_df.iloc[0]
print("Reference:", sample_turn["text"])
asr_results = transcribe_multi("dummy_path.wav", segment_id=sample_turn["segment_id"], fallback_texts={
    "whisper": sample_turn["text"],
    "crisperwhisper": sample_turn["text"],
    "vosk": sample_turn["text"]
})
print("Whisper transcribed:", asr_results["whisper"]["transcript"])
print("WER:", compute_wer(sample_turn["text"], asr_results["whisper"]["transcript"]))

In [ ]:
# 5. Gate Features Extraction
# Features: confidence, +-2 window, 3-way agreement, Zipf frequency, prosody
# (accent_cluster and speaker_id are strictly excluded from classifier)
features_df = build_features([asr_results["whisper"]])
print("Extracted features shape:", features_df.shape)
features_df.head()

In [ ]:
# 6. Gate Training (LightGBM with genuine recall >= 90% constraint)
# Grouped 5-fold cross-validation on training split
print("Training LightGBM gate and evaluating leave-one-accent-out...")